# Day 1 - Exercise 01: Understanding GenAI vs Agentic AI

## Real-World Scenario: Customer Service Bot

**Context:** You work at an e-commerce company. The support team receives hundreds of queries daily about order status, returns, and product information.

In this exercise, you'll build TWO versions of a customer service assistant:
1. **GenAI Version** — Simple Q&A (no tools, no memory)
2. **Agentic AI Version** — Can look up orders, check inventory, and remember conversation context

By the end, you'll clearly see the difference between reactive GenAI and proactive Agentic AI.

---

## Setup

In [3]:
%pip install python-dotenv langchain-anthropic langchain-core langchain

Defaulting to user installation because normal site-packages is not writeable
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached anyio-4.14.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import os
# Initialize Claude
llm = ChatAnthropic(
    api_key=os.environ.get("KEY"),
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    base_url="https://llmgw-wp.tekstac.com/v1/chat/completions"
)
print("Setup complete!")

Setup complete!


---
## Part 1: GenAI Version (Reactive, No Tools, No Memory)

This version can only answer based on what it knows. It cannot:
- Look up actual order information
- Remember what the customer said earlier
- Take any real actions

In [7]:
# GenAI Customer Service Bot - Simple Q&A

def genai_support_bot(question: str) -> str:
    """Simple GenAI bot - no memory, no tools, just Q&A"""
    messages = [
        SystemMessage(content="You are a helpful customer service assistant for ShopEasy e-commerce."),
        HumanMessage(content=question)
    ]
    response = llm.invoke(messages)
    return response.content

# Test the GenAI bot
print("=" * 60)
print("GenAI Bot Response:")
print("=" * 60)
print(genai_support_bot("What is the status of my order #12345?"))

GenAI Bot Response:


APIConnectionError: Connection error.

###  Problem: The GenAI bot CANNOT actually look up order #12345!

It can only give generic responses. Let's see another limitation...

In [ ]:
# Testing memory limitation
print("Turn 1:")
print(genai_support_bot("My name is Sarah and my order number is #12345"))

print("\n" + "=" * 60)
print("Turn 2:")
print(genai_support_bot("What is my name and order number?"))

###  Problem: It forgot everything from Turn 1!

Each call is independent — no memory between turns.

---

## Part 2: Agentic AI Version (Tools + Memory + Reasoning)

Now let's build a proper agent that can:
- ✅ Look up actual order information (Tool Use)
- ✅ Remember conversation context (Memory)
- ✅ Decide what action to take (Reasoning)

In [ ]:
# Simulated database (in real-world, this would be your actual database)
ORDERS_DB = {
    "12345": {
        "customer": "Sarah Johnson",
        "status": "Shipped",
        "items": ["Wireless Headphones", "Phone Case"],
        "tracking": "TRK-789456123",
        "delivery_date": "June 26, 2026"
    },
    "67890": {
        "customer": "Mike Chen",
        "status": "Processing",
        "items": ["Laptop Stand", "USB-C Hub"],
        "tracking": None,
        "delivery_date": "Pending"
    }
}

INVENTORY_DB = {
    "Wireless Headphones": {"stock": 45, "price": 79.99},
    "Phone Case": {"stock": 120, "price": 19.99},
    "Laptop Stand": {"stock": 0, "price": 49.99},  # Out of stock!
    "USB-C Hub": {"stock": 30, "price": 39.99}
}

print("Databases loaded!")

In [ ]:
from langchain_core.tools import tool

# Define tools that the agent can use

@tool
def lookup_order(order_id: str) -> str:
    """Look up order details by order ID. Returns order status, items, and tracking info."""
    order_id = order_id.replace("#", "").strip()
    if order_id in ORDERS_DB:
        order = ORDERS_DB[order_id]
        return f"""Order #{order_id}:
- Customer: {order['customer']}
- Status: {order['status']}
- Items: {', '.join(order['items'])}
- Tracking: {order['tracking'] or 'Not yet available'}
- Expected Delivery: {order['delivery_date']}"""
    return f"Order #{order_id} not found in our system."

@tool
def check_inventory(product_name: str) -> str:
    """Check if a product is in stock and get its price."""
    for name, info in INVENTORY_DB.items():
        if product_name.lower() in name.lower():
            status = "In Stock" if info['stock'] > 0 else "Out of Stock"
            return f"{name}: {status} ({info['stock']} units) - ${info['price']}"
    return f"Product '{product_name}' not found."

@tool
def initiate_return(order_id: str, reason: str) -> str:
    """Initiate a return request for an order."""
    order_id = order_id.replace("#", "").strip()
    if order_id in ORDERS_DB:
        return f"Return request initiated for Order #{order_id}. Reason: {reason}. Return label will be emailed within 24 hours."
    return f"Cannot initiate return - Order #{order_id} not found."

tools = [lookup_order, check_inventory, initiate_return]
print("Tools defined:", [t.name for t in tools])

In [ ]:
from langchain.agents import create_agent

# Create the Agentic AI bot with tools and memory

system_prompt = """You are a helpful customer service agent for ShopEasy e-commerce.
    
You have access to tools to:
- Look up order status and tracking
- Check product inventory
- Initiate returns

Always use the appropriate tool to get real information. Be friendly and helpful.
If you need to look something up, do it before responding."""

# Create the agent using LangChain 1.x API
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

# Memory store - now uses HumanMessage and AIMessage format
chat_history = []

def agentic_support_bot(user_input: str) -> str:
    """Agentic bot with tools and memory"""
    global chat_history
    
    # Build messages list: system + chat history + current input
    messages = chat_history + [HumanMessage(content=user_input)]
    
    # Invoke the agent with messages
    result = agent.invoke({"messages": messages})
    
    # Extract the response (it's the last message in the result)
    response_text = result["messages"][-1].content
    
    # Update memory
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response_text))
    
    return response_text

print("Agentic bot ready!")

In [ ]:
# Test 1: Order lookup (Tool Use)
print("=" * 60)
print("Customer: What is the status of my order #12345?")
print("=" * 60)
response = agentic_support_bot("What is the status of my order #12345?")
print("\nAgent Response:")
print(response)

### ✅ The agent actually looked up the real order data!

In [ ]:
# Test 2: Memory - Follow-up question
print("=" * 60)
print("Customer: When will it arrive?")
print("=" * 60)
response = agentic_support_bot("When will it arrive?")
print("\nAgent Response:")
print(response)

### ✅ The agent remembered we were talking about order #12345!

In [ ]:
# Test 3: Multi-step reasoning - Complex request
print("=" * 60)
print("Customer: I want to return the headphones. They don't fit well.")
print("=" * 60)
response = agentic_support_bot("I want to return the headphones. They don't fit well.")
print("\nAgent Response:")
print(response)

### ✅ The agent:
1. Remembered we were discussing order #12345
2. Initiated the return with the reason provided
3. Took real action!

---

##  Your Turn: Practice Exercises

### Exercise 1.1: Test the Inventory Tool
Ask the agent about product availability.

In [ ]:
# TODO: Ask the agent if the Laptop Stand is in stock
# Hint: The agent should use the check_inventory tool

response = agentic_support_bot("Is the Laptop Stand available?")
print(response)

### Exercise 1.2: Add a New Tool

Create a tool that allows the agent to apply a discount code.

In [ ]:
# TODO: Create a discount code tool
DISCOUNT_CODES = {
    "SAVE10": {"discount": 10, "min_order": 50},
    "WELCOME20": {"discount": 20, "min_order": 100},
}

@tool
def validate_discount_code(code: str) -> str:
    """Validate a discount code and return the discount details."""
    # YOUR CODE HERE
    # 1. Check if code exists in DISCOUNT_CODES
    # 2. Return discount details or "Invalid code" message
    pass

# After creating the tool, add it to the tools list and recreate the agent

### Exercise 1.3: Compare Side-by-Side

Fill in the comparison table based on what you observed:

| Feature | GenAI Bot | Agentic Bot |
|---------|-----------|-------------|
| Can look up real order data? | ❌ No | ✅ Yes |
| Remembers previous messages? | _____ | _____ |
| Can take actions (returns)? | _____ | _____ |
| Decides what tool to use? | _____ | _____ |
| Provides accurate info? | _____ | _____ |

---
##  Key Takeaways

1. **GenAI** is reactive — it responds to prompts but cannot take real actions
2. **Agentic AI** is proactive — it can reason, use tools, and maintain memory
3. The **Agent Loop** enables multi-step reasoning: Think → Act → Observe → Repeat
4. **Tools** give agents real-world capabilities (databases, APIs, etc.)
5. **Memory** allows agents to maintain context across a conversation

---

**Next Exercise →** Building Your First LangChain Agent from Scratch